# Validation of Rotational-Speed Labels

## Methodology

CSV recordings in the `data` directory whose filenames identify one of four motor conditions (`healthy`, `misalignment`, `rear_ball`, or `front_ball`) are included. Recordings marked `_ENV_` or `_SF_` are excluded because environmental contamination, sensor failure, or sensor drift may invalidate direct comparison with the commanded operating condition. The nominal rotational speed is parsed from the filename token of the form `<value>rpm` and treated as the expected speed for that recording.

The measured speed is derived from the sampled `pg_rpm` pulse train following the procedure used in `RPM_calculation.ipynb`. Let $s[k] \in \{0,1\}$ denote the sampled pulse state and let $t_k$ denote the timestamp of the $k$-th detected rising edge. A rising edge is identified when $s[k-1]=0$ and $s[k]=1$. Because the motor driver produces $N=6$ rising-edge pulses per mechanical revolution, each revolution-averaged speed estimate is calculated from edges separated by six pulse intervals:

$$
T_{\mathrm{rev},i}=t_{i+N}-t_i, \qquad \mathrm{RPM}_i=\frac{60}{T_{\mathrm{rev},i}}.
$$

Timestamps recorded in microseconds are converted to seconds before calculating the revolution periods. Non-positive periods are excluded. For each file, the average RPM is defined as the arithmetic mean of the resulting $\mathrm{RPM}_i$ sequence; the minimum and maximum values describe its observed range. The relative difference between average and expected speed is reported as an absolute percentage:

$$
\mathrm{Difference}\;(\%)=100\left|\frac{\overline{\mathrm{RPM}}-\mathrm{RPM}_{\mathrm{expected}}}{\mathrm{RPM}_{\mathrm{expected}}}\right|.
$$

Recordings with a finite relative difference not exceeding 10% are marked with a green check; all other recordings are marked with a red cross.

In [41]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

PULSES_PER_REV = 6
RPM_TOLERANCE_PERCENT = 10.0
GROUPS = ("healthy", "misalignment", "rear_ball", "front_ball")
EXCLUDED_QUALITY_MARKERS = ("_ENV_", "_SF_")
RPM_PATTERN = re.compile(r"_(\d+(?:\.\d+)?)rpm(?:_|$)", re.IGNORECASE)

data_candidates = (Path("../data"), Path("data"))
DATA_DIR = next((path.resolve() for path in data_candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the data directory.")

rows = []
for csv_path in sorted(DATA_DIR.glob("*.csv")):
    if any(marker in csv_path.name.upper() for marker in EXCLUDED_QUALITY_MARKERS):
        continue

    group = next((name for name in GROUPS if name in csv_path.stem.lower()), None)
    rpm_match = RPM_PATTERN.search(csv_path.stem)
    if group is None or rpm_match is None:
        continue

    expected_rpm = float(rpm_match.group(1))
    recording = pd.read_csv(csv_path, usecols=["t_us", "pg_rpm"])
    pulse_signal = pd.to_numeric(recording["pg_rpm"], errors="coerce")
    timestamps_s = pd.to_numeric(recording["t_us"], errors="coerce") * 1e-6
    rising_edges = pulse_signal.eq(1) & pulse_signal.shift(1).eq(0)
    pulse_times_s = timestamps_s.loc[rising_edges].dropna().to_numpy()

    revolution_periods_s = (
        pulse_times_s[PULSES_PER_REV:] - pulse_times_s[:-PULSES_PER_REV]
    )
    valid_periods = revolution_periods_s[
        np.isfinite(revolution_periods_s) & (revolution_periods_s > 0)
    ]
    if valid_periods.size == 0:
        rpm_values = np.array([np.nan])
    else:
        rpm_values = 60.0 / valid_periods

    average_rpm = float(np.mean(rpm_values))
    difference_percent = (
        abs(average_rpm - expected_rpm) / expected_rpm * 100.0
        if expected_rpm != 0
        else np.nan
    )
    within_tolerance = (
        np.isfinite(difference_percent)
        and difference_percent <= RPM_TOLERANCE_PERCENT
    )
    rows.append(
        {
            "Group": group,
            "File": csv_path.name,
            "Expected RPM": expected_rpm,
            "Average RPM": average_rpm,
            "Difference (%)": difference_percent,
            "Within 10%": "✓" if within_tolerance else "✗",
            "Min RPM": float(np.min(rpm_values)),
            "Max RPM": float(np.max(rpm_values)),
        }
    )

rpm_validation = pd.DataFrame(rows)
rpm_validation["Group"] = pd.Categorical(
    rpm_validation["Group"], categories=GROUPS, ordered=True
)
rpm_validation = (
    rpm_validation.sort_values(["Group", "File"])
    .set_index(["Group", "File"])
    .round(2)
)


def color_tolerance_status(value):
    color = "#198754" if value == "✓" else "#dc3545"
    return f"color: {color}; font-weight: bold"


styled_validation = rpm_validation.style.map(
    color_tolerance_status, subset=["Within 10%"]
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(styled_validation)

## RPM Validation Experiment Matrix

### Methodology

The experiment matrix summarizes the coverage and outcome of rotational-speed validation across motor units. The motor unit and experiment replicate are decoded from the numeric identifier immediately following the health-state label in each filename. Consistent with the dataset naming convention, the leading digit or digits identify the motor unit, while the final digit identifies the replicate. The commanded rotational speed and mechanical-load current are extracted from the `<speed>rpm` and `<load-current>mA` filename fields, respectively. Recordings marked `_ENV_` or `_SF_` are excluded before matrix construction.

A separate matrix is constructed for each motor unit. Matrix columns represent the expected rotational speed, and rows represent the applied load-current setting. The current settings correspond approximately to 15%, 30%, 50%, 75%, and 100% mechanical load for 19, 38, 64, 96, and 128 mA, respectively.

Each matrix cell aggregates all eligible recordings available for the corresponding motor, expected-speed, and load combination, including different health states, power sources, and experimental replicates. A green check indicates that every available recording in the cell has a finite relative RPM difference not exceeding 10%. A red cross indicates that at least one recording exceeds this limit or lacks a valid RPM estimate. An em dash denotes a speed-load combination for which no recording is available.

In [44]:
from IPython.display import Markdown

EXPERIMENT_PATTERN = re.compile(
    r"^analize_(?:healthy|misalignment|rear_ball|front_ball)(\d+)",
    re.IGNORECASE,
)
LOAD_PATTERN = re.compile(r"_(\d+)mA(?:_|$)", re.IGNORECASE)
LOAD_LABELS = {
    19: "19 mA (~15%)",
    38: "38 mA (~30%)",
    64: "64 mA (~50%)",
    96: "96 mA (~75%)",
    128: "128 mA (~100%)",
}

matrix_data = rpm_validation.reset_index()
experiment_ids = matrix_data["File"].str.extract(EXPERIMENT_PATTERN, expand=False)
loads_ma = matrix_data["File"].str.extract(LOAD_PATTERN, expand=False)
if experiment_ids.isna().any() or loads_ma.isna().any():
    invalid_files = matrix_data.loc[
        experiment_ids.isna() | loads_ma.isna(), "File"
    ].tolist()
    raise ValueError(f"Could not decode experiment metadata: {invalid_files}")

matrix_data["Motor unit"] = experiment_ids.str[:-1].astype(int) + 1
matrix_data["Experiment"] = experiment_ids.str[-1].astype(int) + 1
matrix_data["Load (mA)"] = loads_ma.astype(int)
motor_units = sorted(matrix_data["Motor unit"].unique())
expected_speeds = sorted(matrix_data["Expected RPM"].astype(int).unique())
load_order = list(LOAD_LABELS)


def aggregate_validation_status(statuses):
    return "✓" if statuses.eq("✓").all() else "✗"


def color_matrix_status(value):
    if value == "✓":
        return "color: #198754; font-weight: bold; font-size: 16px"
    if value == "✗":
        return "color: #dc3545; font-weight: bold; font-size: 16px"
    return "color: #6c757d"


experiment_matrices = {}
for motor_unit in motor_units:
    motor_data = matrix_data.loc[matrix_data["Motor unit"] == motor_unit]
    matrix = (
        motor_data.groupby(["Load (mA)", "Expected RPM"], observed=True)[
            "Within 10%"
        ]
        .agg(aggregate_validation_status)
        .unstack()
        .reindex(index=load_order, columns=expected_speeds)
        .fillna("—")
    )
    matrix.index = [LOAD_LABELS[load] for load in matrix.index]
    matrix.index.name = "Load"
    matrix.columns = matrix.columns.astype(int)
    matrix.columns.name = "Expected RPM"
    experiment_matrices[motor_unit] = matrix

    display(Markdown(f"### Motor unit {motor_unit}"))
    display(
        matrix.style.map(color_matrix_status)
        .set_properties(**{"text-align": "center"})
        .set_table_styles(
            [{"selector": "th", "props": [("text-align", "center")]}]
        )
    )

### Motor unit 2

Expected RPM,500,1000,1500,2000,2500,3000
Load,,,,,,
19 mA (~15%),✓,✓,✓,✓,—,—
38 mA (~30%),✓,✓,✓,✓,—,—
64 mA (~50%),✓,✓,✓,✓,—,—
96 mA (~75%),✓,✓,✓,✓,✓,✓
128 mA (~100%),✓,✓,✓,✓,✓,✓


### Motor unit 3

Expected RPM,500,1000,1500,2000,2500,3000
Load,,,,,,
19 mA (~15%),✓,✓,✓,—,—,—
38 mA (~30%),✓,✓,✓,—,—,—
64 mA (~50%),✓,✓,✓,—,—,—
96 mA (~75%),✓,✓,✓,✓,✓,✓
128 mA (~100%),✓,✓,✓,✓,✓,✓


### Motor unit 4

Expected RPM,500,1000,1500,2000,2500,3000
Load,,,,,,
19 mA (~15%),✓,✓,✓,—,—,—
38 mA (~30%),✓,✓,✓,—,—,—
64 mA (~50%),✓,✓,✓,—,—,—
96 mA (~75%),✓,✓,✓,✓,✓,✓
128 mA (~100%),✓,✓,✓,✓,✓,✓


## Empirical RPM Range and Uncertainty

### Methodology

Rotational speed was estimated from PG-signal rising edges. Because six pulses correspond to one mechanical revolution, the $i$-th revolution period and rotational speed were calculated as

$$
T_i=t_{i+6}-t_i, \qquad \mathrm{RPM}_i=\frac{60}{T_i}.
$$

For each recording, the observed range $R$ and empirical uncertainty $U_{\mathrm{emp}}$ were defined by

$$
R=\mathrm{RPM}_{\max}-\mathrm{RPM}_{\min},
$$

$$
U_{\mathrm{emp}}=\max\left(
\overline{\mathrm{RPM}}-\mathrm{RPM}_{\min},\,
\mathrm{RPM}_{\max}-\overline{\mathrm{RPM}}
\right).
$$

The temporal resolution of the sampled signal was $T_s=1\,\mathrm{ms}$. The corresponding rotational-speed bounds were evaluated at the measured mean revolution period $\overline{T}=60/\overline{\mathrm{RPM}}$:

$$
\mathrm{RPM}_{\mathrm{low}}=\frac{60}{\overline{T}+T_s}, \qquad
\mathrm{RPM}_{\mathrm{high}}=\frac{60}{\overline{T}-T_s}.
$$

The uncertainty attributable to temporal resolution was taken as

$$
U_{\mathrm{pulse}}=\max\left(
\overline{\mathrm{RPM}}-\mathrm{RPM}_{\mathrm{low}},\,
\mathrm{RPM}_{\mathrm{high}}-\overline{\mathrm{RPM}}
\right).
$$

A measurement was classified as consistent with the resolution limit when $U_{\mathrm{emp}}\leq U_{\mathrm{pulse}}$. For measurements exceeding this limit, the relative excess was calculated as

$$
E=100\left(\frac{U_{\mathrm{emp}}}{U_{\mathrm{pulse}}}-1\right)\%.
$$

All uncertainty quantities were derived from measured RPM values and the sampling resolution; nominal RPM was not used.

In [50]:
TIMESTAMP_RESOLUTION_S = 1e-3

uncertainty_data = rpm_validation.reset_index()
experiment_ids = uncertainty_data["File"].str.extract(
    r"^analize_(?:healthy|misalignment|rear_ball|front_ball)(\d+)",
    expand=False,
)
if experiment_ids.isna().any():
    invalid_files = uncertainty_data.loc[experiment_ids.isna(), "File"].tolist()
    raise ValueError(f"Could not decode motor unit: {invalid_files}")

uncertainty_data["Motor unit"] = experiment_ids.str[:-1].astype(int) + 1
uncertainty_data["Ref RPM"] = uncertainty_data["Expected RPM"].astype(int)
uncertainty_data["Load (mA)"] = pd.to_numeric(
    uncertainty_data["File"].str.extract(
        r"_(\d+)mA(?:_|$)", expand=False
    ),
    errors="raise",
).astype(int)

average_rpm = uncertainty_data["Average RPM"]
observed_range_rpm = uncertainty_data["Max RPM"] - uncertainty_data["Min RPM"]
lower_deviation_rpm = average_rpm - uncertainty_data["Min RPM"]
upper_deviation_rpm = uncertainty_data["Max RPM"] - average_rpm
empirical_uncertainty_rpm = np.maximum(lower_deviation_rpm, upper_deviation_rpm)

revolution_period_s = 60.0 / average_rpm
valid_resolution = (
    np.isfinite(revolution_period_s)
    & (revolution_period_s > TIMESTAMP_RESOLUTION_S)
)
resolution_lower_rpm = pd.Series(
    np.where(
        valid_resolution,
        60.0 / (revolution_period_s + TIMESTAMP_RESOLUTION_S),
        np.nan,
    ),
    index=uncertainty_data.index,
)
resolution_upper_rpm = pd.Series(
    np.where(
        valid_resolution,
        60.0 / (revolution_period_s - TIMESTAMP_RESOLUTION_S),
        np.nan,
    ),
    index=uncertainty_data.index,
)
pulse_resolution_limit_rpm = np.maximum(
    average_rpm - resolution_lower_rpm,
    resolution_upper_rpm - average_rpm,
)
within_resolution_limit = (
    np.isfinite(empirical_uncertainty_rpm)
    & np.isfinite(pulse_resolution_limit_rpm)
    & (empirical_uncertainty_rpm <= pulse_resolution_limit_rpm)
)
valid_exceedance = (
    np.isfinite(empirical_uncertainty_rpm)
    & np.isfinite(pulse_resolution_limit_rpm)
    & (pulse_resolution_limit_rpm > 0)
    & ~within_resolution_limit
)
exceeds_limit_percent = pd.Series(
    np.nan, index=uncertainty_data.index, dtype=float
)
exceeds_limit_percent.loc[valid_exceedance] = (
    empirical_uncertainty_rpm.loc[valid_exceedance]
    / pulse_resolution_limit_rpm.loc[valid_exceedance]
    - 1.0
) * 100.0

rpm_uncertainty = uncertainty_data[
    [
        "Group",
        "File",
        "Motor unit",
        "Ref RPM",
        "Load (mA)",
        "Average RPM",
        "Min RPM",
        "Max RPM",
    ]
].copy()
rpm_uncertainty["Observed range (RPM)"] = observed_range_rpm
rpm_uncertainty["Empirical uncertainty (RPM)"] = empirical_uncertainty_rpm
rpm_uncertainty["Pulse-resolution limit (RPM)"] = pulse_resolution_limit_rpm
rpm_uncertainty["Exceeds limit (%)"] = exceeds_limit_percent
rpm_uncertainty["Within uncertainty"] = np.where(within_resolution_limit, "✓", "✗")
rpm_uncertainty = (
    rpm_uncertainty[
        [
            "Group",
            "File",
            "Motor unit",
            "Ref RPM",
            "Load (mA)",
            "Average RPM",
            "Min RPM",
            "Max RPM",
            "Observed range (RPM)",
            "Empirical uncertainty (RPM)",
            "Pulse-resolution limit (RPM)",
            "Exceeds limit (%)",
            "Within uncertainty",
        ]
    ]
    .sort_values(["Group", "Motor unit", "Ref RPM", "Load (mA)", "File"])
    .set_index(["Group", "File"])
    .round(2)
)


def color_uncertainty_status(value):
    color = "#198754" if value == "✓" else "#dc3545"
    return f"color: {color}; font-weight: bold"


styled_uncertainty = (
    rpm_uncertainty.style.map(
        color_uncertainty_status, subset=["Within uncertainty"]
    ).format({"Exceeds limit (%)": "{:.2f}%"}, na_rep="—")
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(styled_uncertainty)